In [93]:
import operator
import random
import json
from deap import base, creator, tools, gp
import pydot

In [24]:
INDICATORS = ["RSI", "SMA"]
NUM_INDICATOR_PARAMS = [5, 10, 14, 20, 30, 100, 200]
SOURCES = ["open", "high", "low", "close", "volume"]
OPS = [operator.gt, operator.ge, operator.eq, operator.ne, operator.lt, operator.le]

In [95]:
# 1.1. 새로운 전용 타입 정의
# BuyType: 매수 로직 내에서만 사용되는 불리언 타입
class BuyType(object): pass
# SellType: 매도 로직 내에서만 사용되는 불리언 타입
class SellType(object): pass

# 1.2. 컨테이너 함수 정의: 두 개의 독립된 타입을 자식으로 받음
def Strategy(buy_logic, sell_logic):
    """최상위 컨테이너: BuyType 로직과 SellType 로직을 독립적으로 받습니다."""
    # 이 함수는 실제 파이썬 코드로 실행될 필요는 없으며, 구조를 정의하는 역할만 합니다.
    return (buy_logic, sell_logic) 


In [96]:
class ConditionManager:
    # ... (ConditionManager 클래스는 이전 코드와 동일)
    def __init__(self, num_conditions=20): # 예시 출력을 위해 조건 수를 줄임
        self.buy_pool = {}
        self.sell_pool = {}
        self.generate_conditions(num_conditions)
    
    def _create_random_condition(self):
        # ... (조건 생성 로직은 이전 코드와 동일)
        indicator = random.choice(INDICATORS)
        source = random.choice(SOURCES)
        param = random.choice(NUM_INDICATOR_PARAMS)
        op_func = random.choice(OPS)
        value = random.uniform(1, 100)
        return {
            "indicator": indicator,
            "source": source,
            "param": param,
            "op": op_func.__name__,
            "value": round(value, 2)
        }

    def generate_conditions(self, n):
        for i in range(n):
            self.buy_pool[f"buy_system{i + 1}"] = self._create_random_condition()
            self.sell_pool[f"sell_system{i + 1}"] = self._create_random_condition()


In [97]:
def _extract_tree_nodes(node):
    """
    주어진 노드를 루트로 하는 서브트리의 모든 노드를 
    깊이 우선 탐색(DFS) 순서로 추출하여 리스트로 반환합니다.
    """
    nodes = [node] # 현재 노드 포함
    
    # Primitive (함수) 노드인 경우 자식 노드를 재귀적으로 탐색
    if isinstance(node, gp.Primitive):
        for arg in node.args:
            # 각 자식 노드에 대해 재귀 호출하여 리스트를 확장합니다.
            nodes.extend(_extract_tree_nodes(arg))
            
    return nodes

def _tree_to_string(tree_nodes):
    """노드 리스트를 (and(sys1, sys2)) 형태의 문자열로 변환합니다."""
    
    # 트리를 문자열로 변환하는 로직은 gp.PrimitiveTree.__str__과 유사하게 
    # 스택 기반의 깊이 우선 탐색을 사용합니다.
    
    if not tree_nodes:
        return "NO_LOGIC"
    
    stack = []
    
    for node in tree_nodes:
        if isinstance(node, gp.Primitive):
            stack.append((node, []))
        # ✨ 수정된 부분: gp.Terminal인지 명확히 확인하고 처리 ✨
        elif isinstance(node, gp.Terminal): 
            stack.append((node.name, []))
        else:
            # BuyType/SellType 타입 객체는 무시하고 건너뛰거나,
            # 트리의 깊이가 1인데 Primitive도 Terminal도 아닌 경우에만 
            # 해당 타입 이름을 반환하도록 합니다.
            # 대부분의 경우, 이 'else' 블록은 불필요한 오류의 근원입니다.
            continue 
            
        # 스택 연산: Primitive의 arity가 충족될 때까지 반복
        while stack and isinstance(stack[-1][0], gp.Primitive) and len(stack[-1][1]) == stack[-1][0].arity:
            prim, args = stack.pop()
            # 함수 이름과 인수를 결합하여 문자열을 만듭니다.
            string = "{}({})".format(prim.name, ", ".join(args))
            if stack:
                stack[-1][1].append(string)
            else:
                return string 
    
    # 트리가 단일 터미널인 경우 (예: ['buy_system1'])
    if stack and isinstance(stack[0][0], str):
        return stack[0][0]
        
    # 트리 내에 로직은 있으나 문자열로 변환이 완료되지 않은 경우 (오류 발생의 주 원인)
    return "INCOMPLETE_LOGIC"

# 평가 및 파싱 함수 (로깅을 위해 print 추가)
def eval_func(individual) -> tuple[float, float]:
    # 유효한 Strategy 트리가 아니면 평가하지 않습니다.
    if individual.root.name != 'Strategy':
        return (-1000, )
    
    # 수익률을 계산하고 적합도를 반환합니다.
    # 로깅을 위해 실제 시뮬레이터 대신 랜덤 값과 낮은 페널티를 사용합니다.
    if len(individual) > 50:
        return (-100,) # 너무 큰 트리에 페널티
    
    profit = random.uniform(10, 50)
    
    return (profit,)

In [ ]:
# 타입 정의 및 PrimitiveSet 설정 (이전 코드와 동일)
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

# 1.3. PrimitiveSet 재정의: Buy와 Sell 로직을 강제 분리
pset = gp.PrimitiveSetTyped("Main", [], object) # 최종 반환 타입은 Strategy의 반환 타입(object)

# A. 최상위 컨테이너 Primitive 등록
# Strategy는 BuyType 1개, SellType 1개를 받아 object 타입(전체 전략)을 반환합니다.
pset.addPrimitive(Strategy, [BuyType, SellType], object)

# B. Buy 로직 전용 Primitive 등록
pset.addPrimitive(operator.and_, [BuyType, BuyType], BuyType)
pset.addPrimitive(operator.or_, [BuyType, BuyType], BuyType)
pset.addPrimitive(operator.not_, [BuyType], BuyType)

# C. Sell 로직 전용 Primitive 등록
pset.addPrimitive(operator.and_, [SellType, SellType], SellType)
pset.addPrimitive(operator.or_, [SellType, SellType], SellType)
pset.addPrimitive(operator.not_, [SellType], SellType)

condition_manager = ConditionManager()

# buy_systemN 터미널은 BuyType을 반환하도록 등록
for term in condition_manager.buy_pool.keys():
    pset.addTerminal(term, BuyType) 
# sell_systemN 터미널은 SellType을 반환하도록 등록
for term in condition_manager.sell_pool.keys():
    pset.addTerminal(term, SellType) 

def parse_gp_tree_to_json(individual):
    """
    TypeError를 수정하고 'vars'를 동적으로 생성하는 최종 파싱 함수.
    """
    # 1. 트리가 유효한지 확인
    if not isinstance(individual, gp.PrimitiveTree) or individual.root.name != 'Strategy':
        return None

    # 2. searchSubtree를 이용해 buy/sell 서브트리의 slice를 가져옴
    buy_subtree_slice = individual.searchSubtree(1)
    buy_tree_content = individual[buy_subtree_slice]
    sell_tree_start_index = buy_subtree_slice.stop
    sell_tree_content = individual[sell_tree_start_index:]

    # 3. 추출한 내용으로 PrimitiveTree 객체를 생성
    buy_tree = gp.PrimitiveTree(buy_tree_content)
    sell_tree = gp.PrimitiveTree(sell_tree_content)

    # 4. 내장 str() 함수를 사용해 문자열로 변환
    buy_op_str = str(buy_tree) if len(buy_tree) > 0 else "NO_LOGIC"
    sell_op_str = str(sell_tree) if len(sell_tree) > 0 else "NO_LOGIC"

    # 5. 각 트리에서 사용된 터미널(system)들을 추출
    buy_systems_obj = {}
    for node in buy_tree:
        if isinstance(node, gp.Terminal) and node.name in condition_manager.buy_pool:
            buy_systems_obj[node.name] = condition_manager.buy_pool[node.name]

    sell_systems_obj = {}
    for node in sell_tree:
        if isinstance(node, gp.Terminal) and node.name in condition_manager.sell_pool:
            sell_systems_obj[node.name] = condition_manager.sell_pool[node.name]

    # ✨ ================== [새로 추가된 부분] ================== ✨
    # 6. 사용된 모든 시스템을 바탕으로 'vars'를 동적으로 생성
    dynamic_vars = {}
    all_systems = {**buy_systems_obj, **sell_systems_obj} # 두 딕셔너리를 하나로 합침
    for system_details in all_systems.values():
        indicator = system_details.get("indicator")
        param = system_details.get("param")
        if indicator and param is not None:
            # ex) "RSI_14" 라는 키 생성
            var_key = f"{indicator}_{param}"
            dynamic_vars[var_key] = param
    # ✨ ======================================================= ✨

    return {
        "vars": dynamic_vars,  # 하드코딩된 값을 동적으로 생성된 딕셔너리로 교체
        "buy_systems": buy_systems_obj,
        "buy_system_op": buy_op_str,
        "sell_systems": sell_systems_obj,
        "sell_system_op": sell_op_str
    }


def generate_strategy_expr(pset, min_depth, max_depth):
    """
    BuyType과 SellType 서브트리를 생성한 후 Strategy 컨테이너로 묶는 함수
    """
    # BuyType 서브트리 생성 (BuyType만 반환하도록 제한)
    buy_expr = gp.genHalfAndHalf(pset, min_depth, max_depth, type_=BuyType)
    # SellType 서브트리 생성 (SellType만 반환하도록 제한)
    sell_expr = gp.genHalfAndHalf(pset, min_depth, max_depth, type_=SellType)
    
    # Strategy(BuyTree, SellTree) 구조를 명시적으로 만듭니다.
    return [pset.primitives[BuyType][0], buy_expr, pset.primitives[SellType][0], sell_expr]

def combined_expr():
    """BuyType 또는 SellType 생성 함수 중 하나를 랜덤으로 호출"""
    if random.random() < 0.5:
        return toolbox.expr_buy()
    else:
        return toolbox.expr_sell()

# Toolbox 설정
toolbox = base.Toolbox()
toolbox.register("expr_buy", gp.genHalfAndHalf, pset=pset, min_=1, max_=3, type_=BuyType)
toolbox.register("expr_sell", gp.genHalfAndHalf, pset=pset, min_=1, max_=3, type_=SellType)

# Strategy 함수 자체를 PrimitiveSet에서 찾습니다.
Strategy_prim = pset.primitives[object][0] 

def strategy_root_initializer():
    """Strategy 노드를 루트로 하는 개체의 구성을 시작합니다."""
    return Strategy_prim

# toolbox.expr 정의 (mutate에서 사용했던 combined_expr과 혼동 방지)
def initial_strategy_generator():
    """Strategy 노드를 루트로 하는 완전한 트리를 생성 시도"""
    # gp.genHalfAndHalf이 object 타입을 목표로 전체 트리를 생성하도록 합니다.
    return gp.genHalfAndHalf(pset, min_=1, max_=4, type_=object)

# ✨ ================== [핵심 수정 부분] ================== ✨
# 1. 타입에 맞는 표현식을 생성하는 새로운 함수를 정의합니다.
#    mutUniform이 이 함수를 호출할 때 교체될 노드의 'type_'를 인자로 넘겨줍니다.
def safe_generate_expr(pset, type_):
    return gp.genHalfAndHalf(pset, min_=1, max_=2, type_=type_)

toolbox.register("expr_init", initial_strategy_generator)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr_init) 
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("expr", combined_expr)
toolbox.register("evaluate", eval_func)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mate", gp.cxOnePoint)
# 2. 위에서 정의한 함수를 expr_mut 라는 이름으로 toolbox에 등록합니다.
toolbox.register("expr_mut", safe_generate_expr, pset=pset)

# 3. mutate 연산자가 expr_mut를 사용하도록 최종 등록합니다.
#    이제 변이가 일어날 때, BuyType 노드는 새로운 BuyType 트리로만,
#    SellType 노드는 새로운 SellType 트리로만 교체됩니다.
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=pset)

/home/tako/Documents/yonghan/GenTradingRuleWithGP/.conda/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/tako/Documents/yonghan/GenTradingRuleWithGP/.conda/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [99]:
def visualize_tree(individual, pset, filename="best_strategy_tree.png"):
    """최적의 GP 트리를 PNG 파일로 저장합니다."""
    
    # gp.graph 함수를 사용하여 노드와 엣지 리스트를 가져옵니다.
    # pset (PrimitiveSet)을 전달하여 노드 이름을 가져옵니다.
    nodes, edges, labels = gp.graph(individual)

    # pydot 그래프 객체 생성
    graph = pydot.Dot(graph_type='graph', bgcolor="#f0f0f0") 
    
    # 노드 추가
    for node in nodes:
        label = labels[node]
        if label in ['and', 'or', 'not']:
            style = "filled"
            fillcolor = "#FFA07A"
        elif label.startswith("buy_system"):
            style = "filled"
            fillcolor = "#F08080"  # 매수 (붉은 계열)
        elif label.startswith("sell_system"):
            style = "filled"
            fillcolor = "#ADD8E6"  # 매도 (하늘색)
        elif label == 'Strategy':
            style = "filled"
            fillcolor = "#CCFFCC" # 컨테이너 (녹색 계열)
        else:
             style = "filled"
             fillcolor = "#FFFFCC"
        
        graph.add_node(pydot.Node(node, label=label, style=style, fillcolor=fillcolor))

    # 엣지 추가
    for edge in edges:
        graph.add_edge(pydot.Edge(edge[0], edge[1]))

    # 파일로 저장
    try:
        # graph.write_png()는 graphviz 실행 파일 경로가 시스템 환경 변수에 등록되어 있어야 합니다.
        graph.write_png(filename)
        print(f"\n✅ 트리 시각화 완료: '{filename}' 파일이 저장되었습니다.")
        print("   (파일을 열어 복잡한 전략 구조를 확인해 보세요.)")
    except FileNotFoundError:
        print(f"\n❌ 트리 시각화 실패: Graphviz 실행 파일을 찾을 수 없습니다.")
        print("   Graphviz 프로그램이 설치되어 있고 시스템 PATH에 등록되어 있는지 확인하세요.")
    except Exception as e:
        print(f"\n❌ 트리 시각화 실패: 오류 발생 - {e}")

In [100]:
# =========================================================================
# 2. 메인 진화 루프 (로깅 기능 추가)
# =========================================================================
if __name__ == "__main__":
    random.seed(42) # 재현성을 위해 시드 고정
    N_POP = 30     # 개체수 감소
    N_GEN = 5      # 세대수 감소

    pop = toolbox.population(n=N_POP)
    
    # 초기 개체군 평가
    print("="*60)
    print("✨ 초기 개체군 (0세대) 평가 시작...")
    fitnesses = toolbox.map(toolbox.evaluate, pop)
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
        print(f"  [Init] Fit: {fit[0]:.2f} | Size: {len(ind):<2} | Tree: {str(ind)}")
    
    # 세대별 진화 루프
    for gen in range(1, N_GEN + 1):
        print("\n" + "="*60)
        print(f"🧬 [세대 {gen}] 진화 시작")

        # 1. 선택 (Selection)
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))
        print(f"  ➡️ 선택 완료: {len(offspring)}개의 개체가 다음 단계로 이동합니다.")

        # 2. 교차 (Crossover)
        print("  🔄 교차 (Crossover) 단계:")
        for i in range(0, len(offspring), 2):
            child1, child2 = offspring[i], offspring[i+1]
            if random.random() < 0.9: # 높은 확률로 교차
                print(f"    - 부모 {i+1} 교차 전: {str(child1)}")
                print(f"    - 부모 {i+2} 교차 전: {str(child2)}")
                
                # 교차 실행
                toolbox.mate(child1, child2)
                
                print(f"    - 자식 {i+1} 교차 후: {str(child1)}")
                print(f"    - 자식 {i+2} 교차 후: {str(child2)}")
                del child1.fitness.values
                del child2.fitness.values
        
        # 3. 변이 (Mutation)
        print("  🦠 변이 (Mutation) 단계:")
        for i, mutant in enumerate(offspring):
            if random.random() < 0.2: # 20% 확률로 변이
                original_tree = str(mutant)
                
                # 변이 실행
                toolbox.mutate(mutant)
                
                print(f"    - 개체 {i+1} 변이 전: {original_tree}")
                print(f"    - 개체 {i+1} 변이 후: {str(mutant)}")
                del mutant.fitness.values
        
        # 4. 새로운 개체군 평가
        print("  📊 새로운 개체군 평가 시작:")
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
            print(f"    - 평가 완료 | Fit: {fit[0]:.2f} | Size: {len(ind):<2} | Tree: {str(ind)}")
        
        # 5. 다음 세대 업데이트
        pop[:] = offspring
        
        # 통계 출력
        fits = [ind.fitness.values[0] for ind in pop]
        length = len(pop)
        mean = sum(fits) / length
        best_fit = max(fits)
        print(f"  ⭐️ 세대 {gen} 통계: 평균 적합도 = {mean:.2f}, 최고 적합도 = {best_fit:.2f}")


    # 최종 결과 출력
    best_ind = tools.selBest(pop, 1)[0]
    final_json_strategy = parse_gp_tree_to_json(best_ind)
    
    print("\n" + "="*60)
    print("🏆 최종 최적 전략")
    print(f"트리 구조: {str(best_ind)}")
    print(f"최고 적합도: {best_ind.fitness.values[0]:.2f}")
    
    if final_json_strategy:
        print("\n[ 최종 JSON 출력 ]")
        print(json.dumps(final_json_strategy, indent=4))
        visualize_tree(best_ind, pset)
    else:
        print("최적 개체가 불완전한 전략(Strategy 노드로 시작하지 않음)이므로 JSON 출력을 건너뜁니다.")

✨ 초기 개체군 (0세대) 평가 시작...
  [Init] Fit: -1000.00 | Size: 3  | Tree: or_('sell_system9', 'sell_system8')
  [Init] Fit: -1000.00 | Size: 3  | Tree: or_('sell_system18', 'sell_system14')
  [Init] Fit: 42.02 | Size: 3  | Tree: Strategy('buy_system7', 'sell_system8')
  [Init] Fit: -1000.00 | Size: 3  | Tree: or_('sell_system18', 'sell_system15')
  [Init] Fit: -1000.00 | Size: 2  | Tree: not_('sell_system6')
  [Init] Fit: -1000.00 | Size: 15 | Tree: or_(and_(and_('buy_system11', 'buy_system4'), and_('buy_system13', 'buy_system4')), or_(or_('buy_system20', 'buy_system9'), and_('buy_system15', 'buy_system18')))
  [Init] Fit: 14.37 | Size: 5  | Tree: Strategy('buy_system20', or_('sell_system3', 'sell_system8'))
  [Init] Fit: -1000.00 | Size: 2  | Tree: not_('sell_system8')
  [Init] Fit: -1000.00 | Size: 3  | Tree: or_('buy_system12', 'buy_system12')
  [Init] Fit: -1000.00 | Size: 4  | Tree: or_(not_('sell_system6'), 'sell_system8')
  [Init] Fit: -1000.00 | Size: 2  | Tree: not_('buy_system18')
  